In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.4 Temperature and decoding

The distribution is fixed; **decoding** is how we turn it into text.
Temperature reshapes the distribution before we sample from it.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
for t in [0, 0.5, 1.0, 1.5]:
    print(f"--- temperature={t} ---")
    print(generate(model, tok, "The ", max_new_tokens=60, temperature=t, seed=0))

Low temperature is confident and repetitive; high temperature is diverse
and chaotic. `top_k` and `top_p` instead clip the unlikely tail before sampling:

In [ ]:
print("top_k=5: ", generate(model, tok, "The ", max_new_tokens=60, top_k=5, seed=0))
print("top_p=0.9:", generate(model, tok, "The ", max_new_tokens=60, top_p=0.9, seed=0))

In [ ]:
# Temperature 0 is greedy argmax, so it is fully deterministic.
a = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
b = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
assert a == b